# PubMedQA LoRA+ on Modal

Primary fine-tuning notebook for running PubMedQA LoRA+ training inside a Modal notebook.

Assumptions:
- one strong GPU such as `A100`, `H100`, or `L40S`
- one persistent Modal Volume mounted at `PUBMEDQA_MODAL_ROOT`
- `HF_TOKEN` available in the Modal environment
- optional `WANDB_API_KEY`, `ADAPTER_HUB_REPO_ID`, and `MERGED_HUB_REPO_ID`


In [ ]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "accelerate",
        "datasets",
        "huggingface_hub",
        "peft",
        "scikit-learn",
        "sentencepiece",
        "transformers",
        "tqdm",
        "wandb",
    ]
)


In [ ]:

import inspect
import json
import os
import random
import warnings
from pathlib import Path
from typing import Any

import numpy as np
import torch
import wandb
from datasets import Dataset, DatasetDict, load_dataset, load_from_disk
from huggingface_hub import login, snapshot_download
from peft import LoraConfig, PeftModel, get_peft_model
from peft.optimizers import create_loraplus_optimizer
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

REPO_ROOT = Path.cwd()
CONFIG_PATH = REPO_ROOT / "configs" / "loraplus_train.json"
DEFAULT_CONFIG = (
    json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    if CONFIG_PATH.exists()
    else {
        "run_name": "pubmedqa-loraplus",
        "base_model_id": "mistralai/Mistral-7B-Instruct-v0.3",
        "dataset_id": "qiaojin/PubMedQA",
        "dataset_config": "pqa_labeled",
        "model_dtype": "auto",
        "train_batch_size": 4,
        "eval_batch_size": 4,
        "gradient_accumulation_steps": 4,
        "learning_rate": 5e-5,
        "loraplus_lr_ratio": 16.0,
        "loraplus_lr_embedding": 1e-6,
        "num_epochs": 3,
        "logging_steps": 10,
        "save_steps": 200,
        "eval_steps": 200,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "target_modules": [
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        "warmup_ratio": 0.03,
        "weight_decay": 0.0,
        "lr_scheduler_type": "cosine",
        "group_by_length": True,
        "max_grad_norm": 1.0,
        "max_length": 1024,
        "test_size": 0.1,
        "seed": 42,
        "mmlu_eval_samples": 50,
        "wandb_project": "pubmedqa-loraplus",
        "hub_token_env": "HF_TOKEN",
        "modal_mount_root": "/vol/pubmedqa-loraplus",
    }
)


def env_int(name: str, default: int) -> int:
    return int(os.getenv(name, str(default)))


def env_float(name: str, default: float) -> float:
    return float(os.getenv(name, str(default)))


if os.getenv("HF_TOKEN"):
    login(
        token=os.environ["HF_TOKEN"],
        add_to_git_credential=False,
    )

VOLUME_ROOT = Path(
    os.getenv(
        "PUBMEDQA_MODAL_ROOT",
        DEFAULT_CONFIG.get("modal_mount_root", "/vol/pubmedqa-loraplus"),
    )
)
RUN_NAME = os.getenv(
    "RUN_NAME",
    DEFAULT_CONFIG.get("run_name", "pubmedqa-loraplus"),
)
BASE_MODEL_DIR = VOLUME_ROOT / "base_model"
DATASET_DIR = VOLUME_ROOT / "dataset"
RUN_DIR = VOLUME_ROOT / "runs" / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
ADAPTER_DIR = RUN_DIR / "adapter"
TOKENIZER_DIR = RUN_DIR / "tokenizer"
MERGED_DIR = RUN_DIR / "merged"
REPORT_DIR = VOLUME_ROOT / "reports"
for path in [
    VOLUME_ROOT,
    BASE_MODEL_DIR,
    DATASET_DIR.parent,
    RUN_DIR,
    CHECKPOINT_DIR,
    REPORT_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

CONFIG = {
    **DEFAULT_CONFIG,
    "run_name": RUN_NAME,
    "base_model_id": os.getenv(
        "BASE_MODEL_ID",
        DEFAULT_CONFIG["base_model_id"],
    ),
    "base_model_path": str(BASE_MODEL_DIR),
    "dataset_id": DEFAULT_CONFIG["dataset_id"],
    "dataset_config": DEFAULT_CONFIG["dataset_config"],
    "dataset_disk_path": str(DATASET_DIR),
    "output_root": str(RUN_DIR),
    "reports_root": str(REPORT_DIR),
    "modal_mount_root": str(VOLUME_ROOT),
    "model_dtype": os.getenv(
        "MODEL_DTYPE",
        DEFAULT_CONFIG.get("model_dtype", "auto"),
    ),
    "train_batch_size": env_int(
        "TRAIN_BATCH_SIZE",
        DEFAULT_CONFIG["train_batch_size"],
    ),
    "eval_batch_size": env_int(
        "EVAL_BATCH_SIZE",
        DEFAULT_CONFIG["eval_batch_size"],
    ),
    "gradient_accumulation_steps": env_int(
        "GRADIENT_ACCUMULATION_STEPS",
        DEFAULT_CONFIG["gradient_accumulation_steps"],
    ),
    "learning_rate": env_float(
        "LEARNING_RATE",
        DEFAULT_CONFIG["learning_rate"],
    ),
    "loraplus_lr_ratio": env_float(
        "LORAPLUS_LR_RATIO",
        DEFAULT_CONFIG["loraplus_lr_ratio"],
    ),
    "loraplus_lr_embedding": env_float(
        "LORAPLUS_LR_EMBEDDING",
        DEFAULT_CONFIG["loraplus_lr_embedding"],
    ),
    "num_epochs": env_int(
        "NUM_EPOCHS",
        DEFAULT_CONFIG["num_epochs"],
    ),
    "logging_steps": env_int(
        "LOGGING_STEPS",
        DEFAULT_CONFIG["logging_steps"],
    ),
    "save_steps": env_int(
        "SAVE_STEPS",
        DEFAULT_CONFIG["save_steps"] ,
    ),
    "eval_steps": env_int(
        "EVAL_STEPS",
        DEFAULT_CONFIG["eval_steps"],
    ),
    "max_length": env_int(
        "MAX_LENGTH",
        DEFAULT_CONFIG["max_length"],
    ),
    "mmlu_eval_samples": env_int(
        "MMLU_EVAL_SAMPLES",
        DEFAULT_CONFIG["mmlu_eval_samples"],
    ),
    "wandb_project": os.getenv(
        "WANDB_PROJECT",
        DEFAULT_CONFIG.get("wandb_project", "pubmedqa-loraplus"),
    ),
    "wandb_run_name": RUN_NAME,
    "adapter_hub_repo_id": os.getenv(
        "ADAPTER_HUB_REPO_ID",
        DEFAULT_CONFIG.get("adapter_hub_repo_id"),
    ),
    "merged_hub_repo_id": os.getenv(
        "MERGED_HUB_REPO_ID",
        DEFAULT_CONFIG.get("merged_hub_repo_id"),
    ),
}

os.environ.setdefault("WANDB_PROJECT", CONFIG["wandb_project"])
CONFIG_SNAPSHOT_PATH = REPORT_DIR / f"{RUN_NAME}_modal_notebook_config.json"
CONFIG_SNAPSHOT_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")

print(
    {
        "repo_root": str(REPO_ROOT),
        "modal_root": str(VOLUME_ROOT),
        "gpu_name": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else "cpu"
        ),
        "device_count": torch.cuda.device_count(),
        "base_model_id": CONFIG["base_model_id"],
        "run_name": RUN_NAME,
    }
)
if not torch.cuda.is_available():
    raise RuntimeError("This notebook is intended to run on a Modal GPU instance.")

In [ ]:

if DATASET_DIR.exists() and any(DATASET_DIR.iterdir()):
    print(f"Using cached dataset at {DATASET_DIR}")
else:
    dataset_source = load_dataset(CONFIG["dataset_id"], CONFIG["dataset_config"])
    dataset_source.save_to_disk(str(DATASET_DIR))
    print(f"Saved dataset to {DATASET_DIR}")


def base_model_is_complete(model_dir: Path) -> bool:
    index_path = model_dir / "model.safetensors.index.json"
    if index_path.exists():
        index_payload = json.loads(index_path.read_text(encoding="utf-8"))
        weight_map = index_payload.get("weight_map", {})
        required_files = sorted(set(weight_map.values()))
        missing_files = [name for name in required_files if not (model_dir / name).exists()]
        if missing_files:
            print("Base model cache is incomplete; missing shards:", missing_files)
            return False
        return True

    fallback_files = [
        model_dir / "consolidated.safetensors",
        model_dir / "model.safetensors",
        model_dir / "pytorch_model.bin",
    ]
    return any(path.exists() for path in fallback_files)


if base_model_is_complete(BASE_MODEL_DIR):
    print(f"Using cached base model at {BASE_MODEL_DIR}")
else:
    snapshot_download(
        repo_id=CONFIG["base_model_id"],
        local_dir=str(BASE_MODEL_DIR),
        token=(
            os.getenv(CONFIG.get("hub_token_env", "HF_TOKEN"))
            or None
        ),
        resume_download=True,
    )
    print(f"Saved base model to {BASE_MODEL_DIR}")

cached_dataset = load_from_disk(str(DATASET_DIR))
train_source = (
    cached_dataset["train"]
    if isinstance(cached_dataset, DatasetDict)
    else cached_dataset
)
dataset = train_source.train_test_split(
    test_size=CONFIG["test_size"],
    seed=CONFIG["seed"],
)
print(dataset)
print(dataset["train"][0])


In [ ]:
import sys

sys.path.insert(0, str(REPO_ROOT))

from src.pubmedqa_loraplus.data import encode_prompt
from src.pubmedqa_loraplus.prompts import build_answer_text
from src.pubmedqa_loraplus.scoring import (
    accuracy,
    choose_label,
    confusion_matrix_counts,
    is_collapsed,
    macro_f1,
    majority_class_rate,
    normalise_candidate_scores,
    prediction_counts,
    reference_counts,
    score_candidate,
    select_subject_rows,
    subject_sample_plan,
    weighted_f1,
)

PUBMEDQA_LABELS = ("yes", "no", "maybe")
MMLU_CHOICES = ("A", "B", "C", "D")
MMLU_MEDICAL_TASKS = (
    "anatomy",
    "clinical_knowledge",
    "college_medicine",
    "medical_genetics",
    "professional_medicine",
    "virology",
)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def resolve_model_dtype(value: str) -> torch.dtype:
    if value == "float32":
        return torch.float32
    if value == "float16":
        return torch.float16
    if value == "bfloat16":
        return torch.bfloat16
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def load_causal_lm(model_path: str | Path) -> Any:
    model_load_kwargs = {
        "trust_remote_code": True,
    }
    model_from_pretrained_params = inspect.signature(
        AutoModelForCausalLM.from_pretrained
    ).parameters
    dtype_argument = (
        "dtype"
        if "dtype" in model_from_pretrained_params
        else "torch_dtype"
    )
    model_load_kwargs[dtype_argument] = resolve_model_dtype(CONFIG["model_dtype"])
    return AutoModelForCausalLM.from_pretrained(
        str(model_path),
        **model_load_kwargs,
    )


def get_wandb_metadata() -> dict[str, Any]:
    run = getattr(wandb, "run", None)
    if run is None:
        return {}
    path = getattr(run, "path", None)
    if isinstance(path, (list, tuple)):
        path = "/".join(str(part) for part in path)
    return {
        "url": getattr(run, "url", None),
        "name": getattr(run, "name", None),
        "id": getattr(run, "id", None),
        "project": getattr(run, "project", None),
        "entity": getattr(run, "entity", None),
        "path": path,
    }


def get_model_card_image_url() -> str | None:
    for env_name in [
        "MODEL_CARD_IMAGE_URL",
        "WANDB_IMAGE_URL",
    ]:
        value = os.getenv(env_name)
        if value:
            return value.strip()
    return None


def get_model_card_image_alt() -> str:
    return os.getenv(
        "MODEL_CARD_IMAGE_ALT",
        "Training and evaluation curves",
    )


def latest_logged_metric(log_history: list[dict[str, Any]], key: str) -> float | None:
    for entry in reversed(log_history):
        if key in entry:
            return float(entry[key])
    return None


def flatten_numeric_metrics(prefix: str, payload: dict[str, Any]) -> dict[str, float]:
    metrics: dict[str, float] = {}
    for key, value in payload.items():
        metric_key = f"{prefix}/{key}" if prefix else str(key)
        if isinstance(value, dict):
            metrics.update(flatten_numeric_metrics(metric_key, value))
        elif isinstance(value, (int, float)) and not isinstance(value, bool):
            metrics[metric_key] = float(value)
    return metrics


def format_metric(value: Any) -> str:
    if value is None:
        return "n/a"
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)):
        return f"{float(value):.4f}"
    return str(value)


def render_markdown_table(headers: list[str], rows: list[list[Any]]) -> str:
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for row in rows:
        lines.append("| " + " | ".join(str(cell) for cell in row) + " |")
    return "\n".join(lines)


def subject_accuracies(subject_results: dict[str, dict[str, int]]) -> dict[str, float]:
    return {
        subject: (
            result["correct"] / result["total"] if result["total"] else 0.0
        )
        for subject, result in subject_results.items()
    }


def flatten_context(context: object) -> str:
    if isinstance(context, dict) and "contexts" in context:
        return " ".join(context["contexts"])
    if isinstance(context, list):
        return " ".join(str(part) for part in context)
    return str(context)


def tokenize_example(
    example: dict[str, Any],
    tokenizer: Any,
) -> dict[str, list[int]]:
    answer = build_answer_text(
        example["final_decision"],
        tokenizer.eos_token or "</s>",
    )
    answer_ids = tokenizer(
        answer,
        add_special_tokens=False,
    )["input_ids"]
    max_prompt_tokens = max(CONFIG["max_length"] - len(answer_ids), 1)
    prompt_ids = encode_prompt(
        tokenizer,
        example["question"],
        flatten_context(example["context"]),
        max_prompt_tokens,
    )
    input_ids = (prompt_ids + answer_ids)[: CONFIG["max_length"]]
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": ([-100] * len(prompt_ids) + answer_ids)[: CONFIG["max_length"]],
    }


class CompletionOnlyCollator:
    def __init__(self, pad_token_id: int):
        self.pad_token_id = pad_token_id

    def __call__(
        self,
        features: list[dict[str, list[int]]],
    ) -> dict[str, torch.Tensor]:
        max_length = max(len(feature["input_ids"]) for feature in features)
        input_ids, attention_mask, labels = [], [], []
        for feature in features:
            padding = max_length - len(feature["input_ids"])
            input_ids.append(feature["input_ids"] + [self.pad_token_id] * padding)
            attention_mask.append(feature["attention_mask"] + [0] * padding)
            labels.append(feature["labels"] + [-100] * padding)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def score_one_candidate(
    model: Any,
    tokenizer: Any,
    prompt_ids: list[int],
    candidate: str,
) -> tuple[float, int]:
    candidate_ids = tokenizer(
        candidate,
        add_special_tokens=False,
    )["input_ids"]
    if not candidate_ids:
        raise ValueError("candidate produced no tokens; cannot score it")
    input_ids = torch.tensor(
        [prompt_ids + candidate_ids],
        dtype=torch.long,
        device=next(model.parameters()).device,
    )
    attention_mask = torch.ones_like(input_ids)
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    log_probs = torch.log_softmax(outputs.logits[0], dim=-1)

    def log_prob_at(position: int, token_id: int) -> float:
        return float(log_probs[position, token_id].item())

    return (
        score_candidate(len(prompt_ids), candidate_ids, log_prob_at),
        len(candidate_ids),
    )


def predict_from_candidates(
    model: Any,
    tokenizer: Any,
    prompt_ids: list[int],
    candidates: dict[str, str],
    order: tuple[str, ...],
) -> tuple[str, bool]:
    raw_scores: dict[str, float] = {}
    token_counts: dict[str, int] = {}
    for label, candidate in candidates.items():
        raw_scores[label], token_counts[label] = score_one_candidate(
            model,
            tokenizer,
            prompt_ids,
            candidate,
        )
    # Compare per-token averages, not sums, so a multi-token label is not
    # penalised for having more tokens than a one-token label.
    scores = normalise_candidate_scores(raw_scores, token_counts)
    return choose_label(scores, order=list(order))


def evaluate_pubmedqa(
    model: Any,
    tokenizer: Any,
    eval_dataset: Dataset,
) -> dict[str, Any]:
    predictions, references = [], []
    tied_predictions = 0
    label_candidates = {
        label: build_answer_text(label, tokenizer.eos_token or "</s>")
        for label in PUBMEDQA_LABELS
    }
    model.eval()
    for example in tqdm(eval_dataset, desc="Evaluating PubMedQA"):
        prompt_ids = encode_prompt(
            tokenizer,
            example["question"],
            flatten_context(example["context"]),
            CONFIG["max_length"],
        )
        prediction, tied = predict_from_candidates(
            model,
            tokenizer,
            prompt_ids,
            label_candidates,
            order=PUBMEDQA_LABELS,
        )
        tied_predictions += int(tied)
        predictions.append(prediction)
        references.append(example["final_decision"].lower())

    matrix = confusion_matrix_counts(references, predictions, PUBMEDQA_LABELS)
    collapsed = is_collapsed(predictions)
    warnings: list[str] = []
    if collapsed:
        warnings.append(
            "All predictions are the same label; the classifier has collapsed "
            "and accuracy equals the majority-class rate."
        )
    if tied_predictions:
        warnings.append(
            f"{tied_predictions} prediction(s) were decided by a tie between candidates."
        )

    return {
        "accuracy": accuracy(references, predictions),
        "macro_f1": macro_f1(matrix, PUBMEDQA_LABELS),
        "weighted_f1": weighted_f1(matrix, PUBMEDQA_LABELS),
        "majority_class_rate": majority_class_rate(references),
        "collapsed": collapsed,
        "tied_predictions": tied_predictions,
        "label_order": list(PUBMEDQA_LABELS),
        "confusion_matrix": matrix,
        "reference_counts": reference_counts(references),
        "prediction_counts": prediction_counts(predictions),
        "num_samples": len(predictions),
        "warnings": warnings,
    }


def evaluate_mmlu_medical(model: Any, tokenizer: Any) -> dict[str, Any]:
    mmlu = load_dataset("cais/mmlu", "all", split="test")
    medical = mmlu.filter(lambda row: row["subject"] in MMLU_MEDICAL_TASKS)

    subject_totals = {subject: 0 for subject in MMLU_MEDICAL_TASKS}
    for subject in medical["subject"]:
        if subject in subject_totals:
            subject_totals[subject] += 1

    # Split the sample budget across subjects. Taking the first N rows of the
    # filtered split spends everything on the first subject in row order
    # (anatomy) and leaves the other five with no samples.
    plan = subject_sample_plan(
        subject_totals,
        CONFIG["mmlu_eval_samples"],
        MMLU_MEDICAL_TASKS,
    )
    selected_rows = select_subject_rows(list(medical["subject"]), plan)

    subject_results = {
        subject: {"correct": 0, "total": 0}
        for subject in MMLU_MEDICAL_TASKS
    }
    model.eval()
    for row_index in tqdm(selected_rows, desc="Evaluating medical MMLU"):
        example = medical[row_index]
        prompt = "The following are multiple choice questions about medical knowledge.\n"
        prompt += f"Question: {example['question']}\n"
        for index, choice in enumerate(example["choices"]):
            prompt += f"{chr(65 + index)}. {choice}\n"
        prompt += "Answer:"
        prompt_ids = tokenizer(
            prompt,
            add_special_tokens=False,
        )["input_ids"][-CONFIG["max_length"]:]
        prediction, _ = predict_from_candidates(
            model,
            tokenizer,
            prompt_ids,
            {choice: f" {choice}" for choice in MMLU_CHOICES},
            order=MMLU_CHOICES,
        )
        subject = example["subject"]
        subject_results[subject]["total"] += 1
        if ord(prediction) - ord("A") == example["answer"]:
            subject_results[subject]["correct"] += 1

    total_correct = sum(result["correct"] for result in subject_results.values())
    total_samples = sum(result["total"] for result in subject_results.values())
    covered_subjects = [
        subject
        for subject in MMLU_MEDICAL_TASKS
        if subject_results[subject]["total"] > 0
    ]
    warnings: list[str] = []
    uncovered = [
        subject
        for subject in MMLU_MEDICAL_TASKS
        if subject_results[subject]["total"] == 0
    ]
    if uncovered:
        warnings.append(
            "No MMLU samples were evaluated for: " + ", ".join(uncovered)
        )
    if len(covered_subjects) == 1:
        warnings.append(
            f"The MMLU figure covers a single subject ({covered_subjects[0]}); "
            "it is not a medical-MMLU average."
        )

    return {
        "overall_accuracy": total_correct / total_samples if total_samples else 0.0,
        "subject_accuracies": subject_accuracies(subject_results),
        "subject_results": subject_results,
        "sample_plan": plan,
        "covered_subjects": covered_subjects,
        "total_samples": total_samples,
        "warnings": warnings,
    }

In [ ]:

class NotebookLoRAPlusTrainer(Trainer):
    def create_optimizer(self):
        if self.optimizer is None:
            self.optimizer = create_loraplus_optimizer(
                self.model,
                torch.optim.AdamW,
                lr=self.args.learning_rate,
                loraplus_lr_ratio=CONFIG["loraplus_lr_ratio"],
                loraplus_lr_embedding=CONFIG["loraplus_lr_embedding"],
                loraplus_weight_decay=self.args.weight_decay,
                betas=(self.args.adam_beta1, self.args.adam_beta2),
                eps=self.args.adam_epsilon,
            )
        return self.optimizer


set_seed(CONFIG["seed"])
tokenizer = AutoTokenizer.from_pretrained(str(BASE_MODEL_DIR))
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

tokenized_dataset = dataset.map(
    lambda example: tokenize_example(example, tokenizer),
    remove_columns=dataset["train"].column_names,
)

model = load_causal_lm(BASE_MODEL_DIR)
model.config.use_cache = False
model = get_peft_model(
    model,
    LoraConfig(
        r=CONFIG["lora_r"],
        lora_alpha=CONFIG["lora_alpha"],
        lora_dropout=CONFIG["lora_dropout"],
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=CONFIG["target_modules"],
    ),
)
model.print_trainable_parameters()

training_kwargs = {
    "output_dir": str(CHECKPOINT_DIR),
    "per_device_train_batch_size": CONFIG["train_batch_size"],
    "per_device_eval_batch_size": CONFIG["eval_batch_size"],
    "gradient_accumulation_steps": CONFIG["gradient_accumulation_steps"],
    "learning_rate": CONFIG["learning_rate"],
    "num_train_epochs": CONFIG["num_epochs"],
    "logging_steps": CONFIG["logging_steps"],
    "save_steps": CONFIG["save_steps"],
    "eval_strategy": "steps",
    "eval_steps": CONFIG["eval_steps"],
    "fp16": resolve_model_dtype(CONFIG["model_dtype"]) == torch.float16,
    "bf16": resolve_model_dtype(CONFIG["model_dtype"]) == torch.bfloat16,
    "gradient_checkpointing": True,
    "ddp_find_unused_parameters": False,
    "remove_unused_columns": False,
    "label_names": ["labels"],
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "save_total_limit": 2,
    "warmup_ratio": CONFIG["warmup_ratio"],
    "weight_decay": CONFIG["weight_decay"],
    "lr_scheduler_type": CONFIG["lr_scheduler_type"],
    "group_by_length": CONFIG.get("group_by_length", True),
    "max_grad_norm": CONFIG["max_grad_norm"],
    "report_to": (
        ["wandb"]
        if os.getenv("WANDB_API_KEY")
        or os.getenv("WANDB_MODE", "").lower() == "offline"
        else []
    ),
    "run_name": RUN_NAME,
    "save_safetensors": True,
}

training_args_params = inspect.signature(TrainingArguments.__init__).parameters
if (
    "eval_strategy" not in training_args_params
    and "evaluation_strategy" in training_args_params
):
    training_kwargs["evaluation_strategy"] = training_kwargs.pop("eval_strategy")
elif "eval_strategy" not in training_args_params:
    training_kwargs.pop("eval_strategy", None)

training_kwargs = {
    key: value
    for key, value in training_kwargs.items()
    if key in training_args_params
}
training_args = TrainingArguments(**training_kwargs)

trainer = NotebookLoRAPlusTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=CompletionOnlyCollator(tokenizer.pad_token_id),
)

train_result = trainer.train(
    resume_from_checkpoint=CONFIG.get("resume_from_checkpoint")
)
trainer.save_state()
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(TOKENIZER_DIR))

# PEFT records the base model exactly as it was loaded. This notebook loads
# it from the Modal volume, so rewrite the reference to the public Hub id
# before the adapter is published, and drop the Trainer bookkeeping pickle.
adapter_config_path = ADAPTER_DIR / "adapter_config.json"
adapter_config = json.loads(adapter_config_path.read_text(encoding="utf-8"))
adapter_config["base_model_name_or_path"] = CONFIG["base_model_id"]
adapter_config_path.write_text(
    json.dumps(adapter_config, indent=2) + "\n",
    encoding="utf-8",
)
(ADAPTER_DIR / "training_args.bin").unlink(missing_ok=True)
training_summary = {
    "best_eval_loss": (
        float(trainer.state.best_metric)
        if trainer.state.best_metric is not None
        else None
    ),
    "latest_train_loss": latest_logged_metric(trainer.state.log_history, "loss"),
    "latest_eval_loss": latest_logged_metric(trainer.state.log_history, "eval_loss"),
    "train_runtime": train_result.metrics.get("train_runtime"),
    "train_samples_per_second": train_result.metrics.get("train_samples_per_second"),
    "train_steps_per_second": train_result.metrics.get("train_steps_per_second"),
    "train_loss": train_result.metrics.get("train_loss"),
}

training_payload = {
    "run_name": RUN_NAME,
    "config": CONFIG,
    "train_examples": len(dataset["train"]),
    "eval_examples": len(dataset["test"]),
    "best_model_checkpoint": trainer.state.best_model_checkpoint,
    "global_step": trainer.state.global_step,
    "train_metrics": dict(train_result.metrics),
    "training_summary": training_summary,
    "wandb": get_wandb_metadata(),
    "artifacts": {
        "checkpoints": str(CHECKPOINT_DIR),
        "adapter": str(ADAPTER_DIR),
        "tokenizer": str(TOKENIZER_DIR),
    },
}
(REPORT_DIR / f"{RUN_NAME}_training.json").write_text(
    json.dumps(training_payload, indent=2),
    encoding="utf-8",
)
(REPORT_DIR / f"{RUN_NAME}_train_history.json").write_text(
    json.dumps(trainer.state.log_history, indent=2),
    encoding="utf-8",
)
training_payload

In [ ]:

eval_base_model = load_causal_lm(BASE_MODEL_DIR)
adapter_model = PeftModel.from_pretrained(
    eval_base_model,
    str(ADAPTER_DIR),
).to("cuda")
adapter_model.eval()

evaluation_payload = {
    "run_name": RUN_NAME,
    "config": CONFIG,
    "artifacts": {
        "adapter": str(ADAPTER_DIR),
        "tokenizer": str(TOKENIZER_DIR),
        "checkpoints": str(CHECKPOINT_DIR),
    },
    "evaluation": {
        "pubmedqa": evaluate_pubmedqa(
            adapter_model,
            tokenizer,
            dataset["test"],
        ),
        "mmlu_medical": evaluate_mmlu_medical(adapter_model, tokenizer),
    },
}
evaluation_payload["wandb"] = get_wandb_metadata()
if getattr(wandb, "run", None) is not None:
    wandb.log(flatten_numeric_metrics("evaluation", evaluation_payload["evaluation"]))
(REPORT_DIR / f"{RUN_NAME}.json").write_text(
    json.dumps(evaluation_payload, indent=2),
    encoding="utf-8",
)
evaluation_payload


In [ ]:

export_base_model = load_causal_lm(BASE_MODEL_DIR)
export_adapter = PeftModel.from_pretrained(
    export_base_model,
    str(ADAPTER_DIR),
)
MERGED_DIR.mkdir(parents=True, exist_ok=True)
merged_model = export_adapter.merge_and_unload()
merged_model.save_pretrained(str(MERGED_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(MERGED_DIR))
export_payload = {
    "adapter": str(ADAPTER_DIR),
    "tokenizer": str(TOKENIZER_DIR),
    "merged": str(MERGED_DIR),
}
(REPORT_DIR / f"{RUN_NAME}_exports.json").write_text(
    json.dumps(export_payload, indent=2),
    encoding="utf-8",
)
export_payload
